<a href="https://colab.research.google.com/github/JediMasterKeith/FUNDAI-Laboratories-BASAN/blob/main/Lab4_Logic_KR_Basan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: Logic and Knowledge Representation

## Fundamentals of Artificial Intelligence

**Name :** Keith Lyndon G. Basan

**Course :** BSCSAI

**Section :** 2A

**Date :** 09/15/2026

**GitHub URL :** https://github.com/JediMasterKeith/FUNDAI-Laboratories-BASAN

## Description
This laboratory uses Python and SymPy to perform truth table generation,
satisfiability checking, theorem proving, and logical deduction.

In [43]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, satisfiable
from itertools import product

In [44]:
P, Q, R = symbols( 'P Q R')

In [45]:
from itertools import product


def print_truth_table(expression, symbol_list):
    header = [str(s) for s in symbol_list] + [str(expression)]
    print(" | ".join(header))
    print("-" * (5 * len(header)))

    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        result = bool(expression.subs(mapping))
        row = [str(v) for v in values] + [str(result)]
        print(" | ".join(row))

    print()

In [46]:
print("Negation: NOT P")
print_truth_table(Not(P), [P])

print("Conjunction: P AND Q")
print_truth_table(And(P, Q), [P, Q])

print("Disjunction: P OR Q")
print_truth_table(Or(P, Q), [P, Q])

print("Implication: P -> Q")
print_truth_table(Implies(P, Q), [P, Q])

print("Biconditional: P <- > Q")
print_truth_table(Equivalent(P, Q), [P, Q])

Negation: NOT P
P | ~P
----------
False | True
True | False

Conjunction: P AND Q
P | Q | P & Q
---------------
False | False | False
False | True | False
True | False | False
True | True | True

Disjunction: P OR Q
P | Q | P | Q
---------------
False | False | False
False | True | True
True | False | True
True | True | True

Implication: P -> Q
P | Q | Implies(P, Q)
---------------
False | False | True
False | True | True
True | False | False
True | True | True

Biconditional: P <- > Q
P | Q | Equivalent(P, Q)
---------------
False | False | True
False | True | False
True | False | False
True | True | True



In [47]:
def is_tautology(expression, symbol_list):
    for values in product([False, True], repeat=len(symbol_list)):
        mapping = dict(zip(symbol_list, values))
        if not bool(expression.subs(mapping)):
            return False
    return True


law_of_excluded_middle = Or(P, Not(P))
contradiction = And(P, Not(P))
simple_implication = Implies(P, Q)

print("P OR NOT P is a tautology:", is_tautology(law_of_excluded_middle, [P]))
print("P AND NOT P is a tautology:", is_tautology(contradiction, [P]))
print("P -> Q is a tautology:", is_tautology(simple_implication, [P, Q]))

P OR NOT P is a tautology: True
P AND NOT P is a tautology: False
P -> Q is a tautology: False


In [48]:
def is_satisfiable(expression):
    return satisfiable(expression) is not False


print("P AND NOT P is satisfiable:", is_satisfiable(And(P, Not(P))))
print("P OR Q is satisfiable:", is_satisfiable(Or(P, Q)))
print("P -> Q is satisfiable:", is_satisfiable(Implies(P, Q)))

P AND NOT P is satisfiable: False
P OR Q is satisfiable: True
P -> Q is satisfiable: True


In [49]:
def to_conjunction(kb):
    if isinstance(kb, list):
        return And(*kb)
    return kb


def kb_entails(kb, conclusion):
    kb_expression = to_conjunction(kb)
    counter_check = And(kb_expression, Not(conclusion))
    return satisfiable(counter_check) is False


def check_entailment(kb, conclusion, label="Query"):
    holds = kb_entails(kb, conclusion)
    kb_expression = to_conjunction(kb)
    counterexample = satisfiable(And(kb_expression, Not(conclusion)))

    print(label)

    if holds:
        print("Result: Entailment holds.")
    else:
        print("Result: Entailment does not hold.")
        print("Counterexample model:", counterexample)

    print("-" * 60)
    return holds

In [50]:
Rain, Wet = symbols('Rain Wet')

kb_rain = [
    Implies(Rain, Wet),
    Rain
]

check_entailment(kb_rain, Wet, "Theorem Proving: Rain example")

Theorem Proving: Rain example
Result: Entailment holds.
------------------------------------------------------------


True

In [51]:
kb_invalid = [
    Implies(Rain, Wet),
    Wet
]

check_entailment(kb_invalid, Rain, "Invalid Inference: Affirming the consequent")

Invalid Inference: Affirming the consequent
Result: Entailment does not hold.
Counterexample model: {Wet: True, Rain: False}
------------------------------------------------------------


False

In [52]:
print("Logical Deduction Rules")
print("=" * 60)

# Modus Ponens
check_entailment(
    [P, Implies(P, Q)],
    Q,
    "Modus Ponens: P, P -> Q, therefore Q"
)

# Modus Tollens
check_entailment(
    [Not(Q), Implies(P, Q)],
    Not(P),
    "Modus Tollens: NOT Q, P -> Q, therefore NOT P"
)

Logical Deduction Rules
Modus Ponens: P, P -> Q, therefore Q
Result: Entailment holds.
------------------------------------------------------------
Modus Tollens: NOT Q, P -> Q, therefore NOT P
Result: Entailment holds.
------------------------------------------------------------


True

##Grounded First-Order Logic Example

Full First-Order Logic includes objects and quantifiers.
For this laboratory, we demonstrate a simple grounded FOL example
by converting FOL atoms into propositional symbols.

English:
- All humans are mortal.
- Socrates is human.
- Therefore, Socrates is mortal.

Grounded propositional form:
- Human_Socrates -> Mortal_Socrates

In [53]:
Human_Socrates, Mortal_Socrates = symbols('Human_Socrates Mortal_Socrates')

kb_socrates = [
    Implies(Human_Socrates, Mortal_Socrates),
    Human_Socrates
]

check_entailment(
    kb_socrates,
    Mortal_Socrates,
    "Grounded FOL: Socrates is mortal"
)

Grounded FOL: Socrates is mortal
Result: Entailment holds.
------------------------------------------------------------


True

In [54]:
def make_human_mortal_kb(constants):
    kb = []
    human = {}
    mortal = {}

    for name in constants:
        h, m = symbols(f'Human_{name} Mortal_{name}')
        human[name] = h
        mortal[name] = m
        kb.append(Implies(h, m))

    return kb, human, mortal

constants = ["Socrates", "Plato"]

kb_people, human, mortal = make_human_mortal_kb(constants)

# Add facts
kb_people.append(human["Socrates"])
kb_people.append(human["Plato"])

# Query: Is Plato mortal?
check_entailment(
    kb_people,
    mortal["Plato"],
    "Grounded FOL with multiple constants: Is Plato mortal?"
)

Grounded FOL with multiple constants: Is Plato mortal?
Result: Entailment holds.
------------------------------------------------------------


True

## Guide Questions and Answers

### 1. What is the difference between syntax and semantics?
**Answer :** Syntax refers to the rules that govern the structure and formation of well-formed expressions or sentences in a logical language. It deals with how symbols are arranged. Semantics, on the other hand, deals with the meaning of these expressions, determining their truth value (true or false) based on the interpretation of the symbols and their relationships.

### 2. Why is 'P -> Q' true when 'P' is false?
**Answer :** The implication 'P -> Q' (read as 'If P, then Q') is defined to be true whenever the premise 'P' is false, regardless of the truth value of the conclusion 'Q'. This is known as *vacuously true* or *material implication*. The only case where 'P -> Q' is false is when 'P' is true AND 'Q' is false. If P is false, the implication doesn't claim anything about Q, so it can't be contradicted, making it true by definition.

### 3. What does it mean for a knowledge base to entail a conclusion?
**Answer :** A knowledge base (KB) entails a conclusion (C) if and only if in every possible model (or interpretation of truth values for all symbols) where all sentences in the KB are true, the conclusion C must also be true. In other words, if the KB is true, then C *logically must be* true. There is no scenario where the KB is true and the conclusion C is false.

### 4. How does theorem proving use satisfiability checking?
**Answer :** Theorem proving often utilizes satisfiability checking through a method called *proof by refutation* (or *reductio ad absurdum*). To prove that a knowledge base (KB) entails a conclusion (C), one assumes the negation of the conclusion (NOT C) is true. Then, a satisfiability checker is used to determine if the combined statement `KB AND (NOT C)` is satisfiable. If `KB AND (NOT C)` is *unsatisfiable* (meaning it leads to a contradiction), then the initial assumption that NOT C is true must be false. Therefore, C must be true whenever KB is true, establishing that KB entails C.

### 5. What is one limitation of propositional logic compared to First-Order Logic?
**Answer :** One significant limitation of propositional logic compared to First-Order Logic (FOL) is its inability to express relationships between objects, properties of objects, or to quantify over collections of objects. Propositional logic treats entire statements as atomic propositions (e.g., 'Socrates is human'), while FOL can break down statements into predicates, objects, and quantifiers (e.g., `Human(Socrates)`, `∀x. (Human(x) → Mortal(x))`). This makes FOL much more expressive, allowing it to represent general rules and complex domains more concisely and naturally than propositional logic, which would require an unmanageably large number of distinct propositional symbols for each specific instance.

## Reflection

### Challenges Encountered
- Understanding the nuances of material implication where 'P -> Q' is true when 'P' is false (vacuously true).
- Distinguishing between satisfiability and tautology, and how they relate to theorem proving.
- Grasping the concept of refutation proof and how it leverages unsatisfiability to demonstrate entailment.

### What I Learned
- The fundamental differences between syntax (structure) and semantics (meaning) in logic.
- How truth tables are constructed and used to evaluate logical expressions.
- The practical application of `satisfiable()` from SymPy for checking satisfiability and performing theorem proving via refutation.
- The distinction between propositional logic and First-Order Logic, particularly concerning expressiveness and the ability to represent quantified statements and object properties. I also learned how to 'ground' First-Order Logic statements into propositional logic for simplified analysis.